<a href="https://colab.research.google.com/github/xKDR/Evaluating-LLMs-and-prompting-strategies-for-legal-text-classification/blob/main/reproducible_research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reproducible Research: LLM Classification of Court Orders

This notebook reproduces the comparisons from `simple_compare.py`: classifying Indian court orders as **Substantive (1)** or **Non-substantive (0)** using multiple LLMs and prompt strategies (plain, fewshot).

- **Data**: SQLite database `llm_classification.db` from [xKDR/Evaluating-LLMs-and-prompting-strategies-for-legal-text-classification](https://github.com/xKDR/Evaluating-LLMs-and-prompting-strategies-for-legal-text-classification) (table `court_orders`: `pdf_id`, `gold_standard`, `pymupdf4llm`).
- **Reproducibility**: Data order is fixed (`ORDER BY pdf_id`); there is no shuffling. All API calls use `temperature=0`, so same inputs give the same outputs.
- **API keys**: Set `GEMINI_API_KEY` below; optionally set `HUGGINGFACE_API_KEY` for Gemma/Qwen models.

In [ ]:
# Install required packages (run once). Run this notebook from the repo root.
%pip install -r https://raw.githubusercontent.com/xKDR/Evaluating-LLMs-and-prompting-strategies-for-legal-text-classification/main/requirements.txt

In [ ]:
# --- Configuration (edit before running) ---
# Set your Gemini API key; optional: HuggingFace token for Gemma/Qwen
GEMINI_API_KEY = ""  # e.g. "AIza..."
HUGGINGFACE_API_KEY = ""  # optional: for HuggingFace Inference Endpoints

# Paths
DB_URL = "https://github.com/xKDR/Evaluating-LLMs-and-prompting-strategies-for-legal-text-classification/raw/main/llm_classification.db"
PROJECT_ROOT = None  # set below
DATA_DIR = None
DB_PATH = None

import os
from pathlib import Path
PROJECT_ROOT = Path(os.getcwd()).resolve()
if not (PROJECT_ROOT / "SRC" / "llm_comparison").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # if running from notebooks/
DATA_DIR = PROJECT_ROOT / "DATA"
DB_PATH = DATA_DIR / "llm_classification.db"

# Ensure API key is in environment so simple_compare imports use it
if GEMINI_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
if HUGGINGFACE_API_KEY:
    os.environ["HUGGINGFACE_API_KEY"] = HUGGINGFACE_API_KEY

print("Project root:", PROJECT_ROOT)
print("DB path:", DB_PATH)
print("GOOGLE_API_KEY set:", bool(os.environ.get("GOOGLE_API_KEY")))

In [ ]:
Reproducibility: we do not shuffle the data (order is fixed by `ORDER BY pdf_id`), and all API calls use `temperature=0`, so no random seed is needed.

In [ ]:
# --- Download SQLite database if not present ---
import urllib.request
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not DB_PATH.exists():
    print("Downloading llm_classification.db from GitHub...")
    urllib.request.urlretrieve(DB_URL, DB_PATH)
    print("Saved to", DB_PATH)
else:
    print("Using existing database:", DB_PATH)

In [ ]:
# --- Load data from SQLite (deterministic order: by pdf_id) ---
import sqlite3
import pandas as pd

def load_data_sqlite(db_path, max_orders=None):
    conn = sqlite3.connect(db_path)
    query = "SELECT pdf_id, gold_standard, pymupdf4llm FROM court_orders WHERE gold_standard IS NOT NULL AND gold_standard != 'Skip' AND pymupdf4llm IS NOT NULL ORDER BY pdf_id"
    if max_orders is not None:
        query += f" LIMIT {max_orders}"
    df = pd.read_sql_query(query, conn)
    conn.close()
    df["gold_standard"] = df["gold_standard"].replace({
        "Non-substantive (Heard)": "Non-substantive",
        "Non-substantive (Not heard)": "Non-substantive",
    })
    return df

# Load all orders (set max_orders=50 for a quick test)
MAX_ORDERS = None  # e.g. 50 for a fast run
df = load_data_sqlite(DB_PATH, max_orders=MAX_ORDERS)
print("Loaded", len(df), "orders")
print("Gold standard distribution:")
print(df["gold_standard"].value_counts())
df.head()

In [ ]:
# --- Import comparison logic from simple_compare ---
# Must run after setting GOOGLE_API_KEY so genai is configured correctly.
import sys
import asyncio
import logging
from datetime import datetime
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Add SRC so "from db_connection" and "from llm_comparison.simple_compare" work
sys.path.insert(0, str(PROJECT_ROOT / "SRC"))

from llm_comparison.simple_compare import (
    MODELS,
    LABEL_MAP,
    warmup_huggingface_endpoints,
    process_orders,
    evaluate_results,
)
from llm_comparison.prompts import PROMPT_MODES

print("MODELS:", list(MODELS.keys()))
print("PROMPT_MODES:", PROMPT_MODES)

In [ ]:
# --- Run LLM comparison (same logic as simple_compare.py) ---
# Warm up HuggingFace endpoints, then process all orders with all models and prompt modes.
asyncio.run(warmup_huggingface_endpoints(MODELS))
results = asyncio.run(process_orders(df, MODELS))
res_df = pd.DataFrame(results)
res_df["classification"] = res_df["classification"].map(LABEL_MAP).fillna("N/A")
res_df["variant"] = res_df["model"] + "_" + res_df["prompt_mode"]
pivoted = res_df.pivot(index="pdf_id", columns="variant", values="classification")
pivoted.columns = [f"{c}_classification" for c in pivoted.columns]
final_df = df[["pdf_id", "gold_standard"]].merge(pivoted, on="pdf_id", how="left")
print("Results shape:", final_df.shape)
final_df.head()

In [ ]:
# --- Evaluate and save results ---
evaluate_results(final_df)
output_path = DATA_DIR / f"simple_results_notebook_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
final_df.to_csv(output_path, index=False)
print("Results saved to", output_path)